# Task 1 — Data Acquisition and Preprocessing

Load and preprocess **real-world TCGA-BRCA multi-omics data** from cBioPortal for integrative genomics analysis.

**Real Datasets (TCGA Breast Cancer - PanCancer Atlas):**
- **Expression**: RNA-seq data (1000 genes x 277 samples)
- **Proteomics**: RPPA protein abundance (208 proteins x 277 samples)
- **SNP/Mutations**: Binary mutation matrix (277 samples x 100 genes)
- **Phenotype**: Molecular subtype and clinical data (277 samples)

All datasets share **matching sample IDs** from the same TCGA-BRCA patients.

## 1. Initialize Project Environment

In [1]:
"""Setup and imports for integrative genomics data acquisition."""
import logging
import sys
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
numpy 2.1.3
pandas 2.2.3


## 2. Define Configuration Parameters

In [2]:
@dataclass
class Task1Config:
    """Configuration for data acquisition pipeline."""

    handle: str = "AndreiCod"
    data_dir: Path = Path("./data")  # Real TCGA-BRCA data
    export_dir: Path = Path("./artifacts")
    download_script: Path = Path("../download_data.py")
    random_seed: int = 42

    def __post_init__(self):
        self.export_dir.mkdir(parents=True, exist_ok=True)
        self.data_dir.mkdir(parents=True, exist_ok=True)

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["data_dir"] = str(info["data_dir"])
        info["export_dir"] = str(info["export_dir"])
        info["download_script"] = str(info["download_script"])
        return info

    def data_exists(self) -> bool:
        """Check if all required data files exist."""
        required_files = [
            "expression_data.csv",
            "snp_data.csv",
            "proteomics_data.csv",
            "phenotypes.csv",
        ]
        return all((self.data_dir / f).exists() for f in required_files)


CONFIG = Task1Config()
CONFIG.describe()

{'handle': 'AndreiCod',
 'data_dir': 'data',
 'export_dir': 'artifacts',
 'download_script': '../download_data.py',
 'random_seed': 42}

## 3. Load or Download Real TCGA-BRCA Data

The data is downloaded from **cBioPortal** (https://www.cbioportal.org/) using the REST API.

**Study**: `brca_tcga_pan_can_atlas_2018` (Breast Invasive Carcinoma, TCGA PanCancer Atlas)

If data already exists locally, we load it directly. Otherwise, run the download script.

In [3]:
def load_or_download_data(
    config: Task1Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load real TCGA-BRCA data from local files, or download if not present.

    Returns:
        Tuple of (expression_df, proteomics_df, snp_df, phenotype_df)
    """
    if config.data_exists():
        logging.info("✅ Data files found locally - loading from disk...")
    else:
        logging.info("⚠️ Data not found - running download script...")
        logging.info(f"   Script: {config.download_script}")

        # Run the download script
        result = subprocess.run(
            [sys.executable, str(config.download_script)],
            cwd=str(config.download_script.parent),
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            logging.error(f"Download failed: {result.stderr}")
            raise RuntimeError("Data download failed")

        logging.info("✅ Download complete!")

    # Load all datasets
    expr_df = pd.read_csv(config.data_dir / "expression_data.csv", index_col=0)
    prot_df = pd.read_csv(config.data_dir / "proteomics_data.csv", index_col=0)
    snp_df = pd.read_csv(config.data_dir / "snp_data.csv", index_col=0)
    pheno_df = pd.read_csv(config.data_dir / "phenotypes.csv", index_col=0)

    return expr_df, prot_df, snp_df, pheno_df


# Load the real TCGA-BRCA data
expr_df, prot_df, snp_df, pheno_df = load_or_download_data(CONFIG)

print(f"\n{'=' * 60}")
print("LOADED TCGA-BRCA REAL-WORLD DATA")
print(f"{'=' * 60}")
print(f"Expression:  {expr_df.shape[0]} genes x {expr_df.shape[1]} samples")
print(f"Proteomics:  {prot_df.shape[0]} proteins x {prot_df.shape[1]} samples")
print(f"SNP/Mutations: {snp_df.shape[0]} samples x {snp_df.shape[1]} genes")
print(f"Phenotypes:  {pheno_df.shape[0]} samples")

02:45:35 | INFO | ✅ Data files found locally - loading from disk...



LOADED TCGA-BRCA REAL-WORLD DATA
Expression:  1000 genes x 277 samples
Proteomics:  208 proteins x 277 samples
SNP/Mutations: 277 samples x 100 genes
Phenotypes:  277 samples


In [4]:
# Preview the data
print("=== EXPRESSION DATA (RNA-seq) ===")
print(f"Top genes: {expr_df.index[:10].tolist()}")
print(expr_df.iloc[:5, :5])

print("\n=== SNP/MUTATION DATA ===")
print(f"Top mutated genes: {snp_df.columns[:10].tolist()}")
print(snp_df.iloc[:5, :5])

print("\n=== PROTEOMICS DATA (RPPA) ===")
print(f"Proteins: {prot_df.index[:10].tolist()}")
print(prot_df.iloc[:5, :5])

print("\n=== PHENOTYPE DATA ===")
print(pheno_df.head(10))

=== EXPRESSION DATA (RNA-seq) ===
Top genes: [1277, 1360, 1278, 2335, 1281, 1447, 1113, 8755, 4318, 4057]
      TCGA-3C-AALI-01  TCGA-3C-AALK-01  TCGA-4H-AAAK-01  TCGA-5T-A9QA-01  \
1277      179261.0000       434557.000      413984.0000          6924.69   
1360          17.9445          120.811           5.5319          1242.15   
1278       98721.6000       236808.000      298457.0000          3464.96   
2335      106743.0000       136341.000      244774.0000          6690.90   
1281       71842.8000       196171.000      239839.0000          2737.97   

      TCGA-A1-A0SF-01  
1277       94679.8000  
1360          94.6772  
1278       68324.4000  
2335       26396.5000  
1281       94907.2000  

=== SNP/MUTATION DATA ===
Top mutated genes: ['PIK3CA', 'TTN', 'TP53', 'MUC16', 'CDH1', 'KMT2C', 'MAP3K1', 'GATA3', 'SYNE1', 'RYR2']
                 PIK3CA  TTN  TP53  MUC16  CDH1
TCGA-3C-AALI-01       0    1     1      0     0
TCGA-3C-AALK-01       1    0     0      0     0
TCGA-4H-AAAK-01

In [5]:
# Phenotype distribution
print("=== PHENOTYPE DISTRIBUTION ===")
print("\nMolecular Subtype:")
print(pheno_df["molecular_subtype"].value_counts())

print("\nDrug Response (phenotype):")
print(pheno_df["phenotype"].value_counts())

print("\nCancer Type:")
print(pheno_df["cancer_type"].value_counts())

=== PHENOTYPE DISTRIBUTION ===

Molecular Subtype:
molecular_subtype
BRCA_LumA      122
BRCA_LumB       62
BRCA_Basal      55
BRCA_Her2       22
BRCA_Normal      9
Name: count, dtype: int64

Drug Response (phenotype):
phenotype
non_responder    222
responder         55
Name: count, dtype: int64

Cancer Type:
cancer_type
Breast Invasive Ductal Carcinoma            215
Breast Invasive Lobular Carcinoma            42
Breast Invasive Carcinoma (NOS)              17
Breast Invasive Mixed Mucinous Carcinoma      2
Metaplastic Breast Cancer                     1
Name: count, dtype: int64


## 4. Data Preprocessing

In [6]:
def preprocess_expression(df: pd.DataFrame, apply_log2: bool = True) -> pd.DataFrame:
    """Apply log2 transformation for RNA-seq data."""
    if apply_log2:
        # Only transform if values are large (raw counts)
        if df.max().max() > 100:
            df = df.apply(lambda x: np.log2(x + 1))
            logging.info("Applied log2(x+1) transformation to expression data")
        else:
            logging.info("Expression data already appears log-transformed, skipping")
    return df


def preprocess_proteomics(df: pd.DataFrame) -> pd.DataFrame:
    """Z-score normalize proteomics data."""
    scaler = StandardScaler()
    normalized = pd.DataFrame(
        scaler.fit_transform(df.T).T, index=df.index, columns=df.columns
    )
    logging.info("Applied Z-score normalization to proteomics data")
    return normalized


# Preprocess data
expr_processed = preprocess_expression(expr_df, apply_log2=True)
prot_processed = preprocess_proteomics(prot_df)

print(
    f"\nExpression range: [{expr_processed.values.min():.2f}, {expr_processed.values.max():.2f}]"
)
print(
    f"Proteomics range: [{prot_processed.values.min():.2f}, {prot_processed.values.max():.2f}]"
)
print(f"SNP range: [{snp_df.values.min()}, {snp_df.values.max()}]")

02:45:35 | INFO | Applied log2(x+1) transformation to expression data
02:45:35 | INFO | Applied Z-score normalization to proteomics data



Expression range: [0.00, 20.81]
Proteomics range: [nan, nan]
SNP range: [0, 1]


## 5. Validate Sample Alignment

In [7]:
def align_datasets(
    snp: pd.DataFrame, expr: pd.DataFrame, prot: pd.DataFrame, pheno: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Ensure all datasets share the same samples."""

    # Find common samples
    common_samples = (
        set(snp.index) & set(expr.columns) & set(prot.columns) & set(pheno.index)
    )
    common_samples = sorted(list(common_samples))

    logging.info(f"Common samples across all datasets: {len(common_samples)}")

    # Align all datasets
    snp_aligned = snp.loc[common_samples]
    expr_aligned = expr[common_samples]
    prot_aligned = prot[common_samples]
    pheno_aligned = pheno.loc[common_samples]

    return snp_aligned, expr_aligned, prot_aligned, pheno_aligned


snp_aligned, expr_aligned, prot_aligned, pheno_aligned = align_datasets(
    snp_df, expr_processed, prot_processed, pheno_df
)

print(f"\n{'=' * 60}")
print("ALIGNED DATASETS")
print(f"{'=' * 60}")
print(f"SNP: {snp_aligned.shape}")
print(f"Expression: {expr_aligned.shape}")
print(f"Proteomics: {prot_aligned.shape}")
print(f"Phenotype: {pheno_aligned.shape}")

02:45:35 | INFO | Common samples across all datasets: 277



ALIGNED DATASETS
SNP: (277, 100)
Expression: (1000, 277)
Proteomics: (208, 277)
Phenotype: (277, 8)


## 6. Validate with Unit Tests

In [8]:
# Assertions to validate data quality
assert (
    snp_aligned.shape[0]
    == expr_aligned.shape[1]
    == prot_aligned.shape[1]
    == pheno_aligned.shape[0]
), "Sample counts must match across datasets"

assert set(snp_aligned.index) == set(expr_aligned.columns), (
    "Sample IDs must match between SNP and expression"
)

assert pheno_aligned["phenotype"].isin(["responder", "non_responder"]).all(), (
    "Phenotype must be binary responder/non_responder"
)

assert (snp_aligned.values >= 0).all() and (snp_aligned.values <= 1).all(), (
    "SNP values must be 0 or 1"
)

# Check for known cancer genes in mutation data
known_cancer_genes = ["TP53", "PIK3CA", "CDH1", "GATA3", "MAP3K1"]
found_genes = [g for g in known_cancer_genes if g in snp_aligned.columns]
print(f"Known cancer genes found in mutation data: {found_genes}")

print("\n[OK] All validation tests passed!")

Known cancer genes found in mutation data: ['TP53', 'PIK3CA', 'CDH1', 'GATA3', 'MAP3K1']

[OK] All validation tests passed!


## 7. Export Results

In [9]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Export all datasets with task1_ prefix for downstream tasks
snp_file = EXPORT_DIR / "task1_snp_data.csv"
expr_file = EXPORT_DIR / "task1_expression_data.csv"
prot_file = EXPORT_DIR / "task1_proteomics_data.csv"
pheno_file = EXPORT_DIR / "task1_phenotypes.csv"

snp_aligned.to_csv(snp_file)
expr_aligned.to_csv(expr_file)
prot_aligned.to_csv(prot_file)
pheno_aligned.to_csv(pheno_file)

# Export summary statistics
summary = {
    "n_samples": len(snp_aligned),
    "n_genes": len(expr_aligned),
    "n_proteins": len(prot_aligned),
    "n_snps": snp_aligned.shape[1],
    "n_responders": (pheno_aligned["phenotype"] == "responder").sum(),
    "n_non_responders": (pheno_aligned["phenotype"] == "non_responder").sum(),
    "data_source": "TCGA-BRCA (cBioPortal)",
    "study_id": "brca_tcga_pan_can_atlas_2018",
}
summary_df = pd.DataFrame([summary])
summary_file = EXPORT_DIR / "task1_data_summary.csv"
summary_df.to_csv(summary_file, index=False)

print(f"[OK] SNP data saved to: {snp_file}")
print(f"[OK] Expression data saved to: {expr_file}")
print(f"[OK] Proteomics data saved to: {prot_file}")
print(f"[OK] Phenotypes saved to: {pheno_file}")
print(f"[OK] Summary saved to: {summary_file}")

[OK] SNP data saved to: artifacts/task1_snp_data.csv
[OK] Expression data saved to: artifacts/task1_expression_data.csv
[OK] Proteomics data saved to: artifacts/task1_proteomics_data.csv
[OK] Phenotypes saved to: artifacts/task1_phenotypes.csv
[OK] Summary saved to: artifacts/task1_data_summary.csv


In [10]:
# Display final summary
print("\n" + "=" * 60)
print("TASK 1 DATA ACQUISITION SUMMARY (REAL TCGA-BRCA DATA)")
print("=" * 60)
print(f"Data Source: TCGA Breast Cancer (PanCancer Atlas)")
print(f"Study ID: brca_tcga_pan_can_atlas_2018")
print(f"API: cBioPortal REST API")
print("-" * 60)
print(f"Samples: {summary['n_samples']} (all with matching IDs)")
print(f"Genes (Expression): {summary['n_genes']}")
print(f"Proteins (RPPA): {summary['n_proteins']}")
print(f"Mutated Genes (SNP): {summary['n_snps']}")
print("-" * 60)
print(
    f"Responders (Basal-like): {summary['n_responders']} ({100 * summary['n_responders'] / summary['n_samples']:.1f}%)"
)
print(
    f"Non-responders (Other): {summary['n_non_responders']} ({100 * summary['n_non_responders'] / summary['n_samples']:.1f}%)"
)
print("=" * 60)


TASK 1 DATA ACQUISITION SUMMARY (REAL TCGA-BRCA DATA)
Data Source: TCGA Breast Cancer (PanCancer Atlas)
Study ID: brca_tcga_pan_can_atlas_2018
API: cBioPortal REST API
------------------------------------------------------------
Samples: 277 (all with matching IDs)
Genes (Expression): 1000
Proteins (RPPA): 208
Mutated Genes (SNP): 100
------------------------------------------------------------
Responders (Basal-like): 55 (19.9%)
Non-responders (Other): 222 (80.1%)
